# Colab GPU Services for Foreign Whispers

Run this notebook in Google Colab Pro (any GPU runtime — L4 or A100 recommended) to host the two GPU-bound services that the local `docker compose` stack normally provides:

| Service | Port | Backed by |
| --- | --- | --- |
| Whisper STT (OpenAI-compatible) | 8000 | `faster-whisper` behind a thin FastAPI |
| Chatterbox TTS | 8020 | `chatterbox-tts-api` |

Each is exposed via its own [cloudflared quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) (zero-config, no auth). At the end the notebook prints a `.env` snippet to paste into the project root on your laptop. After that, `docker compose --profile cpu up -d` on the Mac runs the API + frontend locally, calling Colab for all GPU work.

**Runtime expectations** — on the L4 the Chatterbox model takes ~60s to download + load on first call; faster-whisper `base` loads in ~10s. Colab Pro keeps this alive for 24h with background execution enabled.

**One restart caveat** — every time the Colab runtime restarts, the trycloudflare URLs change. You re-paste the printed `.env` snippet and `docker compose up -d` on the Mac to pick them up. There is no persistent URL on the free tunnel tier.

## 0 — Sanity-check the GPU

In [ ]:
!nvidia-smi -L

## 1 — Install dependencies

Two pip installs (whisper + chatterbox API server) plus the cloudflared binary. ~2 minutes.

In [ ]:
# chatterbox-tts-api transitively pulls fastapi + uvicorn at the versions
# its own pins demand. We do NOT pin them ourselves — Colab's preinstalled
# google-adk / torchvision / cupy / jax may complain about the resulting
# graph, but those packages aren't in the dubbing pipeline; the warnings
# are noise. The one observable side-effect is a torch downgrade
# (torchvision becomes unhappy) — also irrelevant to us, since we don't
# call torchvision.
%pip -q install faster-whisper python-multipart chatterbox-tts-api

In [ ]:
import os, urllib.request, stat

CLOUDFLARED = '/usr/local/bin/cloudflared'
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CLOUDFLARED,
    )
    os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)
!cloudflared --version

## 2 — Whisper server (`/v1/audio/transcriptions`)

A minimal OpenAI-compatible endpoint backed by `faster-whisper`. The API client (`api/src/inference/whisper_remote.py`) POSTs `multipart/form-data` with a `file` field and expects a JSON body shaped like Whisper's `verbose_json`: `{language, segments: [{start, end, text}, ...], text}`.

We expose only the endpoint the API actually calls — keeping this server tiny means it boots in seconds and there's nothing to misconfigure.

In [ ]:
%%writefile /content/whisper_server.py
import tempfile, pathlib
from fastapi import FastAPI, File, Form, UploadFile
from faster_whisper import WhisperModel

MODEL_NAME = 'base'  # bump to 'small'/'medium' for higher quality
model = WhisperModel(MODEL_NAME, device='cuda', compute_type='float16')
app = FastAPI(title='faster-whisper shim')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_NAME}

@app.post('/v1/audio/transcriptions')
async def transcribe(file: UploadFile = File(...), response_format: str = Form('verbose_json')):
    suffix = pathlib.Path(file.filename or 'audio.wav').suffix or '.wav'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await file.read())
        path = tmp.name
    segments, info = model.transcribe(path, beam_size=5, word_timestamps=False)
    segs = [
        {'id': i, 'start': s.start, 'end': s.end, 'text': s.text}
        for i, s in enumerate(segments)
    ]
    return {
        'language': info.language,
        'duration': info.duration,
        'segments': segs,
        'text': ' '.join(s['text'].strip() for s in segs),
    }


## 3 — Launch both servers

Each gets its own log file under `/content/`; tail them with `!tail -n 50 /content/whisper.log` if anything looks off.

In [ ]:
import subprocess, time, os, signal

# Kill any prior runs (idempotent across cell re-execs)
!pkill -f 'whisper_server' 2>/dev/null; pkill -f 'chatterbox_tts_api' 2>/dev/null; pkill cloudflared 2>/dev/null; sleep 1

whisper_log = open('/content/whisper.log', 'wb')
whisper_proc = subprocess.Popen(
    ['uvicorn', 'whisper_server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', stdout=whisper_log, stderr=subprocess.STDOUT,
)

tts_log = open('/content/tts.log', 'wb')
# chatterbox-tts-api ships an entrypoint that boots its FastAPI app on $PORT.
tts_env = {**os.environ, 'PORT': '8020', 'DEVICE': 'cuda', 'DEFAULT_MODEL': 'multilingual'}
tts_proc = subprocess.Popen(
    ['python', '-m', 'chatterbox_tts_api'],
    env=tts_env, stdout=tts_log, stderr=subprocess.STDOUT,
)
print('Started: whisper pid', whisper_proc.pid, '/ tts pid', tts_proc.pid)

In [ ]:
import time, urllib.request, urllib.error

def wait_for(url, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                if r.status < 500:
                    return True
        except (urllib.error.URLError, ConnectionResetError):
            pass
        time.sleep(2)
    return False

print('whisper /health:', wait_for('http://127.0.0.1:8000/health'))
# Chatterbox can take a minute on first start while it pulls the model.
print('chatterbox /health:', wait_for('http://127.0.0.1:8020/health', timeout=600))

## 4 — Open two cloudflared quick tunnels

Each tunnel prints its public URL on stderr; we parse it out of the log.

In [ ]:
import re, subprocess, time, pathlib

URL_RE = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')

def open_tunnel(port: int, log_path: str) -> str:
    log = open(log_path, 'wb')
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}', '--no-autoupdate'],
        stdout=log, stderr=subprocess.STDOUT,
    )
    deadline = time.time() + 60
    while time.time() < deadline:
        text = pathlib.Path(log_path).read_text(errors='ignore')
        m = URL_RE.search(text)
        if m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError(f'cloudflared did not publish a URL within 60s — see {log_path}')

WHISPER_URL = open_tunnel(8000, '/content/cf_whisper.log')
TTS_URL     = open_tunnel(8020, '/content/cf_tts.log')
print('Whisper:   ', WHISPER_URL)
print('Chatterbox:', TTS_URL)

## 5 — Smoke-test through the tunnel

If both calls return 200, the whole local-Mac → cloudflared → Colab path is wired correctly.

In [ ]:
import urllib.request, json

with urllib.request.urlopen(f'{WHISPER_URL}/health', timeout=15) as r:
    print('whisper:', r.status, r.read().decode())

import json, urllib.request
req = urllib.request.Request(
    f'{TTS_URL}/v1/audio/speech',
    data=json.dumps({'input': 'hola, esto es una prueba', 'response_format': 'wav'}).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=120) as r:
    audio = r.read()
    print('chatterbox:', r.status, f'wav bytes: {len(audio)}')
    pathlib_path = '/content/probe.wav'
    open(pathlib_path, 'wb').write(audio)

## 6 — `.env` snippet for your laptop

Copy the block below into `<repo>/.env`, then on the Mac run `docker compose --profile cpu up -d` (or restart the api container if it's already up). The local API will route Whisper + Chatterbox calls to Colab over HTTPS.

In [ ]:
print('# ── paste into <repo>/.env ──')
print(f'CHATTERBOX_API_URL={TTS_URL}')
print(f'FW_WHISPER_API_URL={WHISPER_URL}')
print('FW_WHISPER_BACKEND=remote')

## 7 — Keep-alive (optional)

Colab Pro permits background execution, but the runtime can still scale down if both the notebook tab is closed AND the runtime is unused. Run this cell to keep the kernel busy with a tail-of-logs loop. Stop it with the square button when you're done.

In [ ]:
import time, datetime
while True:
    print(datetime.datetime.utcnow().isoformat(), '— alive; whisper+tts up')
    time.sleep(300)